In [1]:
import happybase
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime
from tqdm import tqdm
from pyspark.sql import Row
spark = (
    SparkSession.builder
    .appName("buildmodel")
    .master("spark://spark-master:7077")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/14 16:44:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/14 16:44:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/14 16:44:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/14 16:44:51 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
crypto = spark.table("cryptopredictions.cryptocurrencysnapshot")

26/01/14 16:49:00 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


In [3]:
index = spark.table("cryptopredictions.indexsnapshot")

In [4]:
import happybase

connection = happybase.Connection(host='hbase')
table = connection.table('crypto_index_aggregates')

rows = []
for key, data in table.scan():
    if b'#1m' in key:
        key_str = key.decode()
        try:
            symbol, ts_str, interval = key_str.split('#')
            timestamp = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S")
        except Exception as e:
            # jeśli key nie pasuje do formatu, pomiń
            continue

        row_dict = {
            'symbol': symbol,
            'timestamp': timestamp,
            'interval': interval
        }

        for col, val in data.items():
            col_name = col.decode()
            try:
                row_dict[col_name] = float(val.decode())
            except ValueError:
                row_dict[col_name] = val.decode()

        rows.append(Row(**row_dict))

df = spark.createDataFrame(rows)

df.show(truncate=False)

26/01/14 16:49:56 WARN TaskSetManager: Stage 0 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
[Stage 0:>                                                          (0 + 1) / 1]

+------+-------------------+--------+----------+---------+---------+---------+
|symbol|timestamp          |interval|ohlc:close|ohlc:high|ohlc:low |ohlc:open|
+------+-------------------+--------+----------+---------+---------+---------+
|BTC   |2025-11-06 21:37:00|1m      |100937.93 |100937.93|100937.93|100937.93|
|BTC   |2025-11-06 21:38:00|1m      |100946.74 |100946.74|100922.79|100922.79|
|BTC   |2025-11-06 21:39:00|1m      |100932.99 |100932.99|100918.83|100930.14|
|BTC   |2025-11-06 21:40:00|1m      |100911.99 |100911.99|100898.96|100898.96|
|BTC   |2025-11-06 21:41:00|1m      |100951.99 |100951.99|100924.54|100924.55|
|BTC   |2025-11-06 21:42:00|1m      |101137.41 |101137.41|100976.02|100976.02|
|BTC   |2025-11-06 21:43:00|1m      |101146.47 |101146.47|101120.14|101137.4 |
|BTC   |2025-11-06 21:44:00|1m      |101016.37 |101090.25|101016.37|101090.25|
|BTC   |2025-11-06 21:45:00|1m      |100963.82 |101000.01|100963.82|101000.01|
|BTC   |2025-11-06 21:46:00|1m      |100942.81 |1009

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col

symbols_to_keep = ["BTC", "NIM", "SNP", "DJI", "SOL", "ETH"]
df_filtered = df.filter(col("symbol").isin(symbols_to_keep))

window = Window.partitionBy("symbol").orderBy("timestamp")

df_filtered = df_filtered.withColumn("next_close", lead("ohlc:close", 1).over(window))

In [6]:
from pyspark.sql.functions import first, col

symbols = ["BTC", "NIM", "SNP", "DJI", "SOL", "ETH"]

# Pivot ceny
df_close = df_filtered.groupBy("timestamp") \
    .pivot("symbol", symbols) \
    .agg(first("ohlc:close"))

# Pivot next_close z nowymi nazwami
df_next = df_filtered.groupBy("timestamp") \
    .pivot("symbol", symbols) \
    .agg(first("next_close"))

# Zmieniamy nazwy kolumn next_close przed joinem
for sym in symbols:
    if sym in df_next.columns:
        df_next = df_next.withColumnRenamed(sym, f"{sym}_next_close")

# Join po timestamp
df_pivot = df_close.join(df_next, on="timestamp", how="inner")

# Konwersja kolumn na double (oprócz timestamp)
for col_name in df_pivot.columns:
    if col_name != "timestamp":
        df_pivot = df_pivot.withColumn(col_name, col(col_name).cast("double"))
        
feature_cols = ["NIM", "SNP", "DJI", "SOL", "ETH"]
label_col = "BTC_next_close"
df_pivot = df_pivot.dropna(subset=feature_cols + [label_col])
# df_pivot.show(5)


26/01/14 16:50:31 WARN TaskSetManager: Stage 1 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 16:50:35 WARN TaskSetManager: Stage 2 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
[Stage 8:>                                                          (0 + 1) / 1]

+-------------------+--------+--------------+----------------+--------------+------+-------+--------------+--------------+----------------+--------------+--------------+--------------+
|          timestamp|     BTC|           NIM|             SNP|           DJI|   SOL|    ETH|BTC_next_close|NIM_next_close|  SNP_next_close|DJI_next_close|SOL_next_close|ETH_next_close|
+-------------------+--------+--------------+----------------+--------------+------+-------+--------------+--------------+----------------+--------------+--------------+--------------+
|2025-11-15 11:41:00|95850.01|22900.58984375|6734.10986328125|47147.48046875| 140.7|3161.97|      95811.32|22900.58984375|6734.10986328125|47147.48046875|        140.64|        3161.3|
|2025-11-15 11:43:00|95838.34|22900.58984375|6734.10986328125|47147.48046875|140.59|3161.35|      95807.61|22900.58984375|6734.10986328125|47147.48046875|        140.49|       3159.35|
|2025-11-15 11:45:00|95728.01|22900.58984375|6734.10986328125|47147.4804687

In [10]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

feature_cols = ["NIM", "SNP", "DJI", "SOL", "ETH"]
label_col = "BTC_next_close"

df_ml = df_pivot.dropna(subset=feature_cols + [label_col])

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df_ml)\
    .select("timestamp", "features", col(label_col).cast("double").alias("label"))

from pyspark.sql.functions import to_timestamp, col

df_ml = df_ml.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"))
)

In [24]:
from pyspark.sql.functions import to_timestamp, col, lit

train_df = df_ml.filter(col("timestamp") < to_timestamp(lit("2025-12-14")))
test_df = df_ml.filter(col("timestamp") >= to_timestamp(lit("2025-12-14")))

In [35]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=200,        # liczba drzew
    maxDepth=8,         # głębokość drzew
    stepSize=0.05,      # learning rate
    subsamplingRate=0.8,
    seed=42
)

model = gbt.fit(train_df)

26/01/14 17:26:53 WARN TaskSetManager: Stage 3220 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:26:54 WARN TaskSetManager: Stage 3221 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:26:56 WARN TaskSetManager: Stage 3227 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:26:57 WARN TaskSetManager: Stage 3228 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:27:41 WARN DAGScheduler: Broadcasting large task binary with size 1010.5 KiB
26/01/14 17:27:42 WARN DAGScheduler: Broadcasting large task binary with size 1009.2 KiB
26/01/14 17:27:42 WARN DAGScheduler: Broadcasting large task binary with size 1009.7 KiB
26/01/14 17:27:42 WARN DAGScheduler: Broadcasting large task binary with size 1010.4 KiB
26/01/14 17:27:42 WARN DAGScheduler: Broadcasting large task binary 

In [36]:
predictions = model.transform(test_df)

predictions.select(
    "timestamp",
    "label",
    "prediction"
).show(10)

26/01/14 17:36:15 WARN TaskSetManager: Stage 8041 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:36:17 WARN TaskSetManager: Stage 8042 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+-------------------+--------+-----------------+
|          timestamp|   label|       prediction|
+-------------------+--------+-----------------+
|2025-12-14 00:01:00|90200.01|90227.49537385952|
|2025-12-14 00:03:00|90152.58|90194.00177410229|
|2025-12-14 00:04:00| 90162.7|90194.00177410229|
|2025-12-14 00:05:00|90150.32|90194.00177410229|
|2025-12-14 00:07:00|90148.75|90157.76121675846|
|2025-12-14 00:09:00|90179.69|90194.00177410229|
|2025-12-14 00:10:00|90179.69|90194.00177410229|
|2025-12-14 00:11:00|90199.53|90194.00177410229|
|2025-12-14 00:12:00|90219.08|90194.00177410229|
|2025-12-14 00:14:00|90262.46|90204.44058479229|
+-------------------+--------+-----------------+
only showing top 10 rows



In [37]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)

26/01/14 17:36:24 WARN TaskSetManager: Stage 8050 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:36:25 WARN TaskSetManager: Stage 8051 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

RMSE: 772.9004080334531


In [38]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mse"
)

mse = evaluator.evaluate(predictions)
print("MSE:", mse)

26/01/14 17:36:30 WARN TaskSetManager: Stage 8059 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:36:31 WARN TaskSetManager: Stage 8060 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

MSE: 597375.0407382783


In [39]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator.evaluate(predictions)
print("R2:", r2)

26/01/14 17:36:36 WARN TaskSetManager: Stage 8068 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:36:37 WARN TaskSetManager: Stage 8069 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

R2: 0.4757359113231875


In [41]:
predictions.write.mode("overwrite").saveAsTable("cryptopredictions.btc_predictions_GBTR_2")

26/01/14 17:36:57 WARN TaskSetManager: Stage 8084 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
26/01/14 17:36:58 WARN TaskSetManager: Stage 8085 contains a task of very large size (7444 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [40]:
model.write().overwrite().save("hdfs://namenode:8020/models/btc_model_GBTR_2")

26/01/14 17:36:54 WARN TaskSetManager: Stage 8081 contains a task of very large size (2648 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [44]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit

model_path = "hdfs://namenode:8020/models/btc_model_GBTR_2"
df_model_info = spark.createDataFrame([{
    "model_name": "GBTR_BTC_2",
    "model_path": model_path,
    "rmse": rmse,
    "mse": mse,
    "r2": r2
}])
df_model_info = df_model_info.withColumn("timestamp", current_timestamp())
df_model_info.show(truncate=False)


+----------+--------------------------------------------+-----------------+------------------+-----------------+--------------------------+
|model_name|model_path                                  |mse              |r2                |rmse             |timestamp                 |
+----------+--------------------------------------------+-----------------+------------------+-----------------+--------------------------+
|GBTR_BTC_2|hdfs://namenode:8020/models/btc_model_GBTR_2|597375.0407382783|0.4757359113231875|772.9004080334531|2026-01-14 17:40:43.694394|
+----------+--------------------------------------------+-----------------+------------------+-----------------+--------------------------+



In [45]:
df_model_info.write.mode("append").option("header", True).csv(
    "hdfs://namenode:8020/models/model_registry_csv/"
)

In [46]:
spark.stop()